In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../Data/traffic_cleaned.parquet')

df.head()

,date_time,temp,rain_1h,snow_1h,clouds_all,fog_mm,wind_speed_ms,flood_mm,holiday,weather_main,weather_description,day_type,traffic_volume
0,2012-10-02 09:00:00,288.280,0.0,0.0,40.0,0.0,2.30,0.0,No Holiday,Clouds,scattered clouds,Weekday,5545.0
1,2012-10-02 10:00:00,289.360,0.0,0.0,75.0,0.0,7.45,0.0,No Holiday,Clouds,broken clouds,Weekday,4516.0
2,2012-10-02 12:00:00,290.130,0.0,0.0,90.0,0.0,1.93,0.0,No Holiday,Clouds,overcast clouds,Weekday,5026.0
3,2012-10-02 13:00:00,282.429,0.0,0.0,75.0,0.0,3.56,0.0,No Holiday,Clouds,broken clouds,Weekday,4918.0
4,2012-10-02 14:00:00,291.720,0.0,0.0,1.0,0.0,2.41,0.0,No Holiday,Clear,sky is clear,Weekday,5181.0


In [2]:
df.shape

(45794, 13)

In [3]:
df.dtypes

date_time              datetime64[ns]
temp                          float64
rain_1h                       float64
snow_1h                       float64
clouds_all                    float64
fog_mm                        float64
wind_speed_ms                 float64
flood_mm                      float64
holiday                        object
weather_main                   object
weather_description            object
day_type                       object
traffic_volume                float64
dtype: object

In [4]:
df['date_time'].dtype

dtype('<M8[ns]')

In [5]:
# Extract time-based features

df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek
df['month'] = df['date_time'].dt.month
df['year'] = df['date_time'].dt.year

In [6]:
df[['date_time', 'hour', 'day_of_week', 'month', 'year']].head()

,date_time,hour,day_of_week,month,year
0,2012-10-02 09:00:00,9,1,10,2012
1,2012-10-02 10:00:00,10,1,10,2012
2,2012-10-02 12:00:00,12,1,10,2012
3,2012-10-02 13:00:00,13,1,10,2012
4,2012-10-02 14:00:00,14,1,10,2012


In [7]:
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

In [8]:
df['is_weekend'].value_counts()

is_weekend
0    32777
1    13017
Name: count, dtype: int64

In [9]:
df['is_holiday'] = (
    df['holiday'] != 'No Holiday'
).astype(int)

In [10]:
df['is_holiday'].value_counts()

is_holiday
0    45736
1       58
Name: count, dtype: int64

In [11]:
df['time_period'] = pd.cut(
    df['hour'],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=[
        'Night',
        'Morning',
        'Afternoon',
        'Evening',
        'Late_Night'
    ]
)

In [12]:
df['time_period'].value_counts()

time_period
Morning       11699
Night         11647
Afternoon      9258
Evening        7467
Late_Night     5723
Name: count, dtype: int64

In [14]:
df['congestion_level'] = pd.qcut(
    df['traffic_volume'],
    q=3,
    labels=[
        'Low',
        'Medium',
        'High'
    ]
)

In [15]:
df['congestion_level'].value_counts()

congestion_level
Low       15266
Medium    15265
High      15263
Name: count, dtype: int64

In [16]:
df.head()

,date_time,temp,rain_1h,snow_1h,clouds_all,fog_mm,wind_speed_ms,flood_mm,holiday,weather_main,...,day_type,traffic_volume,hour,day_of_week,month,year,is_weekend,is_holiday,time_period,congestion_level
0,2012-10-02 09:00:00,288.280,0.0,0.0,40.0,0.0,2.30,0.0,No Holiday,Clouds,...,Weekday,5545.0,9,1,10,2012,0,0,Morning,High
1,2012-10-02 10:00:00,289.360,0.0,0.0,75.0,0.0,7.45,0.0,No Holiday,Clouds,...,Weekday,4516.0,10,1,10,2012,0,0,Morning,Medium
2,2012-10-02 12:00:00,290.130,0.0,0.0,90.0,0.0,1.93,0.0,No Holiday,Clouds,...,Weekday,5026.0,12,1,10,2012,0,0,Afternoon,High
3,2012-10-02 13:00:00,282.429,0.0,0.0,75.0,0.0,3.56,0.0,No Holiday,Clouds,...,Weekday,4918.0,13,1,10,2012,0,0,Afternoon,High
4,2012-10-02 14:00:00,291.720,0.0,0.0,1.0,0.0,2.41,0.0,No Holiday,Clear,...,Weekday,5181.0,14,1,10,2012,0,0,Afternoon,High


In [17]:
df.shape

(45794, 21)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45794 entries, 0 to 45793
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date_time            45794 non-null  datetime64[ns]
 1   temp                 45794 non-null  float64       
 2   rain_1h              45794 non-null  float64       
 3   snow_1h              45794 non-null  float64       
 4   clouds_all           45794 non-null  float64       
 5   fog_mm               45794 non-null  float64       
 6   wind_speed_ms        45794 non-null  float64       
 7   flood_mm             45794 non-null  float64       
 8   holiday              45794 non-null  object        
 9   weather_main         45794 non-null  object        
 10  weather_description  45794 non-null  object        
 11  day_type             45794 non-null  object        
 12  traffic_volume       45794 non-null  float64       
 13  hour                 45794 non-

In [19]:
new_features = [
    'hour',
    'day_of_week',
    'month',
    'year',
    'is_weekend',
    'is_holiday',
    'time_period',
    'congestion_level'
]

df[new_features].head(10)

,hour,day_of_week,month,year,is_weekend,is_holiday,time_period,congestion_level
0,9,1,10,2012,0,0,Morning,High
1,10,1,10,2012,0,0,Morning,Medium
2,12,1,10,2012,0,0,Afternoon,High
3,13,1,10,2012,0,0,Afternoon,High
4,14,1,10,2012,0,0,Afternoon,High
5,15,1,10,2012,0,0,Afternoon,High
6,17,1,10,2012,0,0,Evening,High
7,18,1,10,2012,0,0,Evening,High
8,19,1,10,2012,0,0,Evening,Medium
9,20,1,10,2012,0,0,Evening,Medium


In [20]:
print(df.shape)
print(df.columns.tolist())

(45794, 21)
['date_time', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'fog_mm', 'wind_speed_ms', 'flood_mm', 'holiday', 'weather_main', 'weather_description', 'day_type', 'traffic_volume', 'hour', 'day_of_week', 'month', 'year', 'is_weekend', 'is_holiday', 'time_period', 'congestion_level']


In [21]:
df[['hour', 'day_of_week', 'month', 'year',
    'is_weekend', 'is_holiday', 'time_period']].head(10)

,hour,day_of_week,month,year,is_weekend,is_holiday,time_period
0,9,1,10,2012,0,0,Morning
1,10,1,10,2012,0,0,Morning
2,12,1,10,2012,0,0,Afternoon
3,13,1,10,2012,0,0,Afternoon
4,14,1,10,2012,0,0,Afternoon
5,15,1,10,2012,0,0,Afternoon
6,17,1,10,2012,0,0,Evening
7,18,1,10,2012,0,0,Evening
8,19,1,10,2012,0,0,Evening
9,20,1,10,2012,0,0,Evening


In [22]:
df[['hour', 'day_of_week', 'month', 'year',
    'is_weekend', 'is_holiday', 'time_period']].head(10)

,hour,day_of_week,month,year,is_weekend,is_holiday,time_period
0,9,1,10,2012,0,0,Morning
1,10,1,10,2012,0,0,Morning
2,12,1,10,2012,0,0,Afternoon
3,13,1,10,2012,0,0,Afternoon
4,14,1,10,2012,0,0,Afternoon
5,15,1,10,2012,0,0,Afternoon
6,17,1,10,2012,0,0,Evening
7,18,1,10,2012,0,0,Evening
8,19,1,10,2012,0,0,Evening
9,20,1,10,2012,0,0,Evening


In [23]:
df.groupby('congestion_level')['traffic_volume'].agg(
    ['min', 'max', 'mean', 'median', 'count']
)

C:\Users\dell\AppData\Local\Temp\ipykernel_22764\3594680220.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('congestion_level')['traffic_volume'].agg(


,min,max,mean,median,count
congestion_level,,,,,
Low,0.0,2197.0,864.878488,721.0,15266
Medium,2198.0,4573.0,3449.374779,3380.0,15265
High,4574.0,7280.0,5470.294110,5380.0,15263


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45794 entries, 0 to 45793
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date_time            45794 non-null  datetime64[ns]
 1   temp                 45794 non-null  float64       
 2   rain_1h              45794 non-null  float64       
 3   snow_1h              45794 non-null  float64       
 4   clouds_all           45794 non-null  float64       
 5   fog_mm               45794 non-null  float64       
 6   wind_speed_ms        45794 non-null  float64       
 7   flood_mm             45794 non-null  float64       
 8   holiday              45794 non-null  object        
 9   weather_main         45794 non-null  object        
 10  weather_description  45794 non-null  object        
 11  day_type             45794 non-null  object        
 12  traffic_volume       45794 non-null  float64       
 13  hour                 45794 non-

In [25]:
df['year'].value_counts().sort_index()

year
2012     2423
2013     8174
2014     4599
2015     4129
2016     8869
2017    10063
2018     7537
Name: count, dtype: int64

In [26]:
print(df['date_time'].min())
print(df['date_time'].max())

2012-10-02 09:00:00
2018-09-30 23:00:00


In [27]:
print("Holiday:", df['holiday'].nunique())
print("Weather Main:", df['weather_main'].nunique())
print("Weather Description:", df['weather_description'].nunique())
print("Time Period:", df['time_period'].nunique())

Holiday: 12
Weather Main: 11
Weather Description: 37
Time Period: 5


In [28]:
print(df['weather_description'].value_counts())

weather_description
sky is clear                           11102
mist                                    5647
overcast clouds                         4854
broken clouds                           4426
scattered clouds                        3268
light rain                              3213
few clouds                              1843
light snow                              1843
Sky is Clear                            1642
moderate rain                           1580
haze                                    1292
light intensity drizzle                 1043
fog                                      870
proximity thunderstorm                   635
drizzle                                  621
heavy snow                               586
heavy intensity rain                     443
snow                                     277
proximity shower rain                    128
thunderstorm                             114
thunderstorm with heavy rain              63
heavy intensity drizzle            

In [29]:
df['weather_description'] = (
    df['weather_description']
    .str.lower()
    .str.strip()
)

In [30]:
df = df.sort_values('date_time').reset_index(drop=True)

In [31]:
df = df.sort_values('date_time').reset_index(drop=True)

In [32]:
features = [
    'temp',
    'rain_1h',
    'snow_1h',
    'clouds_all',
    'fog_mm',
    'wind_speed_ms',
    'flood_mm',
    'hour',
    'day_of_week',
    'month',
    'year',
    'is_weekend',
    'is_holiday',
    'holiday',
    'weather_description',
    'time_period'
]

In [33]:
target_reg = 'traffic_volume'

In [34]:
target_cls = 'congestion_level'

In [35]:
df.groupby('year')['date_time'].agg(['min', 'max', 'count'])

,min,max,count
year,,,
2012,2012-10-02 09:00:00,2012-12-31 23:00:00,2423
2013,2013-01-01 00:00:00,2013-12-31 23:00:00,8174
2014,2014-01-01 00:00:00,2014-08-08 01:00:00,4599
2015,2015-06-11 20:00:00,2015-12-31 21:00:00,4129
2016,2016-01-01 00:00:00,2016-12-31 23:00:00,8869
2017,2017-01-01 00:00:00,2017-12-31 23:00:00,10063
2018,2018-01-01 00:00:00,2018-09-30 23:00:00,7537


In [36]:
df['weather_description'] = (
    df['weather_description']
    .str.lower()
    .str.strip()
)

In [37]:
df = df.sort_values('date_time').reset_index(drop=True)

In [38]:
df.groupby('year')['date_time'].agg(['min', 'max', 'count'])

,min,max,count
year,,,
2012,2012-10-02 09:00:00,2012-12-31 23:00:00,2423
2013,2013-01-01 00:00:00,2013-12-31 23:00:00,8174
2014,2014-01-01 00:00:00,2014-08-08 01:00:00,4599
2015,2015-06-11 20:00:00,2015-12-31 21:00:00,4129
2016,2016-01-01 00:00:00,2016-12-31 23:00:00,8869
2017,2017-01-01 00:00:00,2017-12-31 23:00:00,10063
2018,2018-01-01 00:00:00,2018-09-30 23:00:00,7537


In [39]:
df.to_parquet(
    '../Data/traffic_feature_engineered.parquet',
    index=False
)